# Exercise 2 — Flood depth for one storm of a parish library — Saint Michael, Barbados

This notebook is the analysis behind the dashboard you build in
**[docs/Barbados/exercise_2.rst](../../docs/Barbados/exercise_2.rst)**. Everything the
`Barbados Hands On 2` dashboard puts on screen — the depth layer, the flooded buildings
and roads, the summary table, the headline card — is computed here first, in plain
code you can read and change.

The question being answered is:

> **In storm 150, how deep is the water on each building and road in Saint Michael?**

**The point of the notebook.** You do not have to follow every detail of the forecast
chain to get value out of this data. What matters is that the data is *open and
ordinary* — a depth array, a table of buildings, a table of roads — and that once you
have it in a dataframe you can turn it into whatever picture answers your question.
Sections 1 to 7 reproduce what the dashboard shows. **Section 8 goes past it**, into
three charts no plugin ships, built from the same three variables. That is the skill
worth taking away: the dashboard is a starting point, not a ceiling.

**Workflow**

1. Open the parish flood-map library and look at one storm
2. Read the storm index: which storms exist and how big they are
3. Load the buildings and roads of Saint Michael
4. Sample the deepest water on each feature's own footprint
5. Group into depth bands
6. Build the impact table — the dashboard's summary tile
7. Map it — the dashboard's impact layer
8. **Extend it**: three visualizations the dashboard does not ship
9. From notebook to dashboard: which plugin does which section

Everything is read from a public S3 bucket. Only the receptor geopackage is downloaded,
and only once.

In [ ]:
!pip install -q "zarr>=3" rasterio pyogrio

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import zarr
from matplotlib.patches import Patch
from zarr.storage import FsspecStore

%matplotlib inline

BUCKET = "https://cog-s3-test-401506828094-us-east-1-an.s3.us-east-1.amazonaws.com"
ROOT = f"{BUCKET}/Barbados_training/Barbados_Tomas_2010_flood_maps_for_IBF"
CYCLE = "20101030.000000"

# One Zarr store per parish; one receptor geopackage for the whole island, so the
# parish selects rows out of it. Both are what the exercise 2 dashboard reads.
PARISH = "BB08_SaintMichael"            # Bridgetown, and a third of the island's exposure
PCODE = PARISH.split("_")[0]
STORE = f"{BUCKET}/Barbados_IBF/Barbados/fim_store_{PARISH}_v1.zarr"
FEATURES = f"{ROOT}/04_ibf_island_deduplicated/barbados_ibf_receptors_full.{CYCLE}.gpkg"

print(f"store {STORE.rsplit('/', 1)[-1]}")

## 1. The storm library

Barbados does not run a hydraulic model when a storm approaches. Each parish has a
**library of 200 synthetic storms**, each with a precomputed maximum flood-depth map,
and the forecast picks from it. That library lives in a **Zarr store** — one array split
into many small files, so you can read one storm without downloading the other 199.

The store carries its own metadata. Read that first: it says where you are, what units
you are in, and what *not* to trust.

In [ ]:
group = zarr.open_group(FsspecStore.from_url(STORE), mode="r")
attrs = dict(group.attrs)

for key, value in attrs.items():
    print(f"{key:20} {value}")

Three things to carry forward:

- **`adm1_name`** — Saint Michael. Each of the eleven parishes has its own store on its
  own window, so `BB08` is 308 × 271 cells and another parish is a different shape.
- **`extent_threshold_m: 0.05`** — the model does not call a cell wet below 5 cm. Reuse
  that number rather than inventing one.
- **`depth_note`** — depth was stored upstream as whole centimetres in a byte, so it
  **saturates at 2.55 m**. A cell reading 2.55 means *at least* 2.55, and no band above
  about 2 m is distinct from the one below it. This is the kind of caveat that should
  travel with a chart, not be discovered by whoever reads it.

`depth` is in **metres**; `transform` and `crs` place each cell — EPSG:4326 at one
arc-second, about 30 m.

In [ ]:
depth = group["depth"]
print(f"depth array: shape {depth.shape}  dtype {depth.dtype}  chunks {depth.chunks}")
print(f"  -> {depth.nbytes / 1e6:.0f} MB in total, "
      f"{np.prod(depth.chunks) * 4 / 1e3:.0f} KB per storm before compression")

STORM = 150                      # try changing this: 0-1 are dry, 199 is the largest
one_storm = np.asarray(depth[STORM], dtype="float32")
one_storm = np.where(np.isfinite(one_storm) & (one_storm > 0), one_storm, 0.0)

wet = one_storm[one_storm >= attrs["extent_threshold_m"]]
print(f"\nstorm {STORM}: {wet.size:,} wet cells, "
      f"max {one_storm.max():.2f} m, mean where wet {wet.mean():.2f} m, "
      f"{int((one_storm >= 2.55).sum()):,} cells at the 2.55 m ceiling")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 7))
shown = np.where(one_storm >= attrs["extent_threshold_m"], one_storm, np.nan)
im = ax.imshow(shown, cmap="Blues", vmin=0, vmax=2.55)
ax.set_title(f"Storm {STORM} — maximum flood depth (m), {attrs['adm1_name']}")
ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=ax, label="depth (m)", shrink=0.7)
plt.show()

## 2. The storm index

Storm 150 is a position, not a name. The store's `index.csv` maps each position to the
scenario it came from (`storm_id`) and its rainfall total (`magnitude_mm`), **sorted by
magnitude** — so stepping through the positions walks severity. That ordering is what
makes the dashboard's storm dropdown mean something.

`extent` is a 0/1 wet mask sitting beside `depth` and is cheaper to read, so the sweep
below uses it to show how the flooded area grows across the library.

In [ ]:
index = pd.read_csv(f"{STORE}/index.csv")
print(f"{len(index)} storms, rainfall totals {index.magnitude_mm.min():g} to "
      f"{index.magnitude_mm.max():g} mm, sorted: {index.magnitude_mm.is_monotonic_increasing}")
display(index.iloc[[0, 1, 100, 150, 199]])

# Cell area from the grid: one arc-second is about 30.9 m north-south, and
# 30.9 * cos(latitude) east-west. Barbados is EPSG:4326, so this correction is needed.
pixel = abs(attrs["transform"][0])
lat = attrs["transform"][5] - 0.5 * attrs["grid_shape"][0] * pixel
metres_ns = pixel * 111_320.0
cell_m2 = metres_ns * metres_ns * np.cos(np.radians(lat))
print(f"one cell is about {metres_ns:.1f} m north-south -> {cell_m2:,.0f} m²")

sample = list(range(0, 200, 10)) + [199]
extent = group["extent"]
area_km2 = [np.asarray(extent[i]).sum() * cell_m2 / 1e6 for i in sample]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(index.magnitude_mm[sample], area_km2, marker="o", ms=3, lw=1.2)
ax.axvline(index.magnitude_mm[STORM], color="crimson", ls="--", lw=1, label=f"storm {STORM}")
ax.set_xlabel("storm rainfall total (mm)"); ax.set_ylabel("flooded area (km²)")
ax.set_title(f"Flooded area across the {attrs['adm1_name']} library (every tenth storm)")
ax.legend(); plt.show()

The first two storms carry a rainfall total of 0.00 mm and flood nothing. They are
offered by the dashboard's storm dropdown like any other position, and they will look
almost dry — which is the honest thing for them to look like. The usable range starts
a little way in.

## 3. The buildings and roads of Saint Michael

The exposure data is one geopackage covering the whole island: **204,727 building
footprints and 22,509 road segments**, each already carrying the four exceedance
probabilities that Exercise 3 works from. This is the *full stock* — nothing has been
filtered out by severity — so a count taken from it is exposure in the ordinary sense.

One file holds all eleven parishes, so the read is filtered on `ADM1_PCODE` before
anything leaves the file. That keeps the other 160,000 geometries from ever being built.

The file is about 80 MB. Reading it straight off HTTPS goes through GDAL's `/vsicurl`,
which range-requests the whole thing once per layer; downloading it once is much faster.

In [ ]:
import os
import tempfile
import urllib.request
from pathlib import Path

import pyogrio


def cached(url):
    """Fetch once into the temp dir and return the local path.

    The transfer lands on a staging name and is moved into place only once it is
    complete, so an interrupted download cannot leave a truncated file cached under
    the real name -- GDAL reports that one as "database disk image is malformed" on
    every later run, and deleting it by hand is the only cure.
    """
    path = Path(tempfile.gettempdir()) / url.rsplit("/", 1)[-1]
    if not path.exists():
        fd, staged = tempfile.mkstemp(dir=path.parent, prefix=f"{path.name}.", suffix=".part")
        os.close(fd)
        try:
            print(f"downloading {path.name} ... (about 80 MB, once)")
            urllib.request.urlretrieve(url, staged)
            os.replace(staged, path)
        finally:
            Path(staged).unlink(missing_ok=True)
    return path


source = cached(FEATURES)
LAYERS = {
    "building": ("buildings_ibf",
                 ["feature_id", "subtype", "critical", "population", "population_per_building",
                  "building_area_m2"]),
    "road": ("roads_ibf", ["feature_id", "road_class", "road_length_m"]),
}

parts = []
for type_, (layer, columns) in LAYERS.items():
    part = pyogrio.read_dataframe(source, layer=layer, columns=columns,
                                  where=f"ADM1_PCODE = '{PCODE}'", use_arrow=True)
    part["type"] = type_
    parts.append(part)
features = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)

# A road has no population and a building no length: fill so every row sums.
for col in ["population_per_building", "building_area_m2", "road_length_m"]:
    features[col] = features[col].fillna(0.0)

print(f"{len(features):,} features in {attrs['adm1_name']}  |  CRS {features.crs.to_string()}")
print(features["type"].value_counts().to_string())
print()
features[["type", "subtype", "population_per_building", "building_area_m2", "road_length_m"]].head(3)

`population_per_building` is a **per-building** estimate: the parish population spread
over its buildings by built-up area. It is not a census count of who lives in each
house, and it should never be presented as one.

The column called plain `population` next to it is a trap worth knowing about — it
repeats the whole *parish* total on every single building. Summing it overstates
exposure by four orders of magnitude, and the result looks entirely plausible until
someone checks it. Whenever you meet a population field, establish whether it is a
total or a disaggregation before you put it in a chart.

In [ ]:
buildings = features[features["type"] == "building"]
print(f"sum of population_per_building : {buildings.population_per_building.sum():>18,.0f}   <- people in {attrs['adm1_name']}")
print(f"sum of population             : {buildings.population.sum():>18,.0f}   <- the parish total, {len(buildings):,} times over")

## 4. Sampling depth onto each feature

We want, for each building, **the deepest water anywhere on its footprint**. That is the
conservative reading, and it matches how the exceedance probabilities in Exercise 3 were
sampled, so the two products stay comparable.

The usual shortcut — burn each feature's id into a grid, then read depths back by id —
does not work here. At 30 m a cell is bigger than most Barbadian buildings, and in
Bridgetown up to twenty-five of them share one cell, so burning ids keeps the last one
and silently drops the rest. The sampling has to allow **many features per cell**.

So build the list of (feature, cell) pairs once:

1. For each feature, take the cells its bounding box covers — a handful each.
2. Keep only the pairs where the feature actually touches the cell. One vectorised
   intersection test, not 65,000 separate ones.
3. For each storm, take the maximum depth per feature over its pairs with
   `np.maximum.at` — a single pass.

Only step 3 depends on the storm, so steps 1 and 2 are paid once and reused for all 200.
That is what makes section 8's sweeps cheap, and it is exactly why the dashboard plugin
caches this step rather than redoing it every time you move the storm slider.

In [ ]:
import shapely

zarr_transform = rasterio.Affine(*attrs["transform"])
rows, cols = attrs["grid_shape"]
features = features.to_crs(attrs["crs"])      # both are already EPSG:4326, but never assume

# 1. The cells each feature's bounding box covers. Row 0 is the top edge and the
#    cell height `e` is negative, so the first row comes from the *maximum* y.
a, _, x0, _, e, y0 = zarr_transform[:6]
minx, miny, maxx, maxy = features.geometry.bounds.to_numpy().T
c0 = np.clip(np.floor((minx - x0) / a).astype(int), 0, cols - 1)
c1 = np.clip(np.floor((maxx - x0) / a).astype(int), 0, cols - 1)
r0 = np.clip(np.floor((maxy - y0) / e).astype(int), 0, rows - 1)
r1 = np.clip(np.floor((miny - y0) / e).astype(int), 0, rows - 1)

n_cells = (r1 - r0 + 1) * (c1 - c0 + 1)
feature_idx = np.repeat(np.arange(len(features)), n_cells)
# Position of each candidate inside its feature's window, unrolled row by row.
within = np.arange(n_cells.sum()) - np.repeat(np.cumsum(n_cells) - n_cells, n_cells)
width = np.repeat(c1 - c0 + 1, n_cells)
row = np.repeat(r0, n_cells) + within // width
col = np.repeat(c0, n_cells) + within % width

# 2. Keep the pairs where the feature really touches the cell.
cell_boxes = shapely.box(x0 + col * a, y0 + (row + 1) * e, x0 + (col + 1) * a, y0 + row * e)
touches = shapely.intersects(features.geometry.to_numpy()[feature_idx], cell_boxes)
feature_idx, cell_idx = feature_idx[touches], (row * cols + col)[touches]

sampled = np.unique(feature_idx)
print(f"{len(feature_idx):,} (feature, cell) pairs kept from {n_cells.sum():,} candidates")
print(f"{len(sampled):,} of {len(features):,} features touch at least one cell "
      f"({100 * len(sampled) / len(features):.1f}%)")
print(f"{len(np.unique(cell_idx)):,} of {rows * cols:,} cells carry a feature; "
      f"the busiest carries {np.bincount(cell_idx).max():,}")

That last line is the check to run whenever a sampling method meets a new grid. A cell
carrying twenty-five buildings is precisely the case an id-burning rasterisation cannot
represent — it keeps one and drops the rest, without complaining.

In [ ]:
def feature_max_depth(storm_index):
    """Deepest water over each feature's own footprint, in metres."""
    layer = np.asarray(depth[storm_index], dtype="float32").ravel()
    # Anything non-finite or negative is "no water", not a depth.
    layer = np.where(np.isfinite(layer) & (layer > 0), layer, 0.0)

    per_feature = np.zeros(len(features), dtype="float32")
    np.maximum.at(per_feature, feature_idx, layer[cell_idx])   # one pass over the pairs
    return per_feature


depths = feature_max_depth(STORM)
print(f"storm {STORM}: {int((depths >= attrs['extent_threshold_m']).sum()):,} features touched by water")
print(f"  deepest single feature: {depths.max():.2f} m")

## 5. Depth bands

Group the depths into bands. These are the bands the dashboard plugins use — round
numbers chosen so each one reads without a legend: roughly ankle, knee, and above head
height. The 0.05 m floor is the store's own `extent_threshold_m`, and the top band starts
at 2 m because the store cannot distinguish anything above 2.55 m.

Assign in **ascending** order and let later assignments win, so each feature ends up in
the deepest band it reaches.

In [ ]:
DEPTH_BANDS = [          # (class value, colour, lower bound in metres)
    (1, "green", 0.05),
    (2, "yellow", 0.30),
    (3, "red", 1.00),
    (4, "purple", 2.00),
]
BAND_LABEL = {1: "0.05 - 0.3 m", 2: "0.3 - 1 m", 3: "1 - 2 m", 4: "2 m or more"}


def classify_depth(values):
    band = np.zeros(len(values), dtype="uint8")     # 0 = dry
    for value, _colour, floor in DEPTH_BANDS:
        band[values >= floor] = value               # deepest band wins
    return band


features["depth_m"] = depths
features["band"] = classify_depth(depths)
features["level"] = features.band.map(BAND_LABEL)

pd.crosstab(features["band"].map({0: "dry", **BAND_LABEL}), features["type"]).reindex(
    ["dry", *BAND_LABEL.values()], fill_value=0)

> ⚠️ **Classify before you round.** If you round the depths to 2 decimals first and *then*
> classify, a feature sitting at 0.0455 m rounds up to 0.05 and gets promoted into the wet
> band. That is a display concern silently changing the answer. Round only what you print.

## 6. The impact table

This is the **summary tile** of the exercise 2 dashboard, computed here. Deepest band
first, so the worst case reads first, with a total across every affected feature. The
population share is against the parish — the sum of `population_per_building` over every
building in Saint Michael, which matches the 77,395 the parish reference table reports.

Change the parish and both the numerator and the denominator change; that is worth saying
out loud the first time someone watches the percentage jump.

In [ ]:
TOTAL_POPULATION = buildings.population_per_building.sum()     # every building in the parish
print(f"{attrs['adm1_name']} population across all buildings: {TOTAL_POPULATION:,.0f}")


def impact_row(name, group):
    in_band = group[group["type"] == "building"]
    return {
        "Depth": name,
        "Buildings": f"{len(in_band):,}",
        "Population": f"{group.population_per_building.sum():,.0f}",
        "Area (m²)": f"{group.building_area_m2.sum():,.0f}",
        "Roads (km)": f"{group.road_length_m.sum() / 1000:,.1f}",
        "% of population": f"{100 * group.population_per_building.sum() / TOTAL_POPULATION:.2f}%",
    }


affected = features[features.band > 0]
# Note: not `rows` -- that name already holds the grid height.
table_rows = [impact_row(BAND_LABEL[v], affected[affected.band == v]) for v, _c, _f in reversed(DEPTH_BANDS)]
table_rows.append(impact_row("TOTAL affected", affected))

pd.DataFrame(table_rows).set_index("Depth")

## 7. Mapping it

And this is the **impact layer** of the exercise 2 dashboard: the affected features,
coloured by depth band, drawn over the depth raster.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
left, bottom, right, top = rasterio.transform.array_bounds(rows, cols, zarr_transform)
ax.imshow(shown, cmap="Blues", vmin=0, vmax=2.55, extent=(left, right, bottom, top))

for value, colour, _floor in DEPTH_BANDS:
    subset = features[features.band == value]
    if not subset.empty:
        subset.plot(ax=ax, color=colour, linewidth=1.2, label=BAND_LABEL[value])

ax.set_title(f"Storm {STORM} — buildings and roads in {attrs['adm1_name']} by flood depth")
ax.set_xticks([]); ax.set_yticks([])
# Buildings draw as polygon collections and roads as lines, so the two share a label
# and neither makes a legend entry matplotlib can reuse. Proxy patches instead.
ax.legend(handles=[Patch(facecolor=c, edgecolor="black", lw=0.4, label=BAND_LABEL[v])
                   for v, c, _f in DEPTH_BANDS],
          loc="lower left", fontsize=8)
plt.show()

Only features that reach a band are drawn. Dry ones are the majority and add nothing a
basemap does not already show — dropping them also shrinks the payload from tens of
thousands of features to a few thousand, which is what keeps the dashboard layer
responsive when the storm changes.

## 8. Extending it — three charts the dashboard does not ship

Sections 1 to 7 reproduce the dashboard. Now the part that matters most.

You have a dataframe with one row per building and road, carrying a depth, a band, a
population, an area, a length, a road class and a `critical` flag. **Nothing stops you
from asking it a different question.** The three charts below took a few lines each, use
no new data, and each answers something the dashboard's three tiles cannot:

| chart | question it answers | who asks it |
|---|---|---|
| critical facilities by depth band | which schools, clinics and emergency buildings go under, and how deep | an operations lead deciding what to evacuate first |
| road kilometres by class and band | is the flooding on residential lanes or on the trunk network | anyone planning access routes |
| all eleven parishes, one storm | where on the island does this scenario hurt most | a national duty officer choosing where to send people |

None of these needed a modelling decision. They needed a `groupby`.

In [ ]:
# --- Chart 1: critical facilities under water -------------------------------
# `critical` flags hospitals, schools, fire and police stations and the like.
# There are few of them and each one matters individually, which is exactly the
# case a percentage hides and a count makes visible.
critical = features[(features["type"] == "building") & (features.critical.astype(bool))]
by_band = critical.band.value_counts().reindex([0, 1, 2, 3, 4], fill_value=0)

fig, ax = plt.subplots(figsize=(7, 3.5))
colours = ["#d9d9d9"] + [c for _v, c, _f in DEPTH_BANDS]
ax.bar(["dry\n(under 5 cm)", *BAND_LABEL.values()], by_band.values, color=colours, edgecolor="black", lw=0.5)
for x, value in enumerate(by_band.values):
    if value:
        ax.text(x, value, f" {value}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("critical buildings")
ax.set_title(f"Storm {STORM}: {int(by_band[1:].sum())} of {len(critical)} critical facilities "
             f"in {attrs['adm1_name']} take water")
plt.show()

critical[critical.band > 0].groupby(["subtype", "level"]).size().unstack(fill_value=0)

In [ ]:
# --- Chart 2: which roads flood, by class -----------------------------------
# A kilometre of trunk road and a kilometre of residential lane are not the same
# thing to someone routing an ambulance. The summary table adds them together;
# this pulls them apart.
roads = features[features["type"] == "road"]
km = (roads[roads.band > 0]
      .assign(km=lambda d: d.road_length_m / 1000)
      .pivot_table(index="road_class", columns="level", values="km", aggfunc="sum", fill_value=0)
      .reindex(columns=list(BAND_LABEL.values()), fill_value=0))
km = km.loc[km.sum(axis=1).sort_values().index]           # worst-affected class on top

fig, ax = plt.subplots(figsize=(8, 4))
left_edge = np.zeros(len(km))
for (value, colour, _floor) in DEPTH_BANDS:
    widths = km[BAND_LABEL[value]].to_numpy()
    ax.barh(km.index, widths, left=left_edge, color=colour, edgecolor="black", lw=0.4,
            label=BAND_LABEL[value])
    left_edge += widths
ax.set_xlabel("flooded road (km)")
ax.set_title(f"Storm {STORM}: flooded road in {attrs['adm1_name']} by class and depth")
ax.legend(fontsize=8)
plt.show()

total_km = roads.road_length_m.sum() / 1000
print(f"{left_edge.sum():,.0f} km of {total_km:,.0f} km of road in the parish is under water "
      f"({100 * left_edge.sum() / total_km:.0f}%)")

In [ ]:
# --- Chart 3: the same storm across all eleven parishes ---------------------
# The dashboard shows one parish at a time, because a plugin answers one request.
# A notebook is under no such constraint: this loops the whole pipeline over every
# parish and ranks them. It takes about ten seconds.
PARISHES = [
    ("BB01_ChristChurch", "Christ Church"), ("BB02_SaintAndrew", "Saint Andrew"),
    ("BB03_SaintGeorge", "Saint George"), ("BB04_SaintJames", "Saint James"),
    ("BB05_SaintJohn", "Saint John"), ("BB06_SaintJoseph", "Saint Joseph"),
    ("BB07_SaintLucy", "Saint Lucy"), ("BB08_SaintMichael", "Saint Michael"),
    ("BB09_SaintPeter", "Saint Peter"), ("BB10_SaintPhilip", "Saint Philip"),
    ("BB11_SaintThomas", "Saint Thomas"),
]


def pair_to_grid(gdf, store_attrs):
    """Sections 4.1 and 4.2, as a function, so any parish can reuse them."""
    n_rows, n_cols = store_attrs["grid_shape"]
    px, _, gx, _, py, gy = rasterio.Affine(*store_attrs["transform"])[:6]
    lo_x, lo_y, hi_x, hi_y = gdf.geometry.bounds.to_numpy().T
    ca = np.clip(np.floor((lo_x - gx) / px).astype(int), 0, n_cols - 1)
    cb = np.clip(np.floor((hi_x - gx) / px).astype(int), 0, n_cols - 1)
    ra = np.clip(np.floor((hi_y - gy) / py).astype(int), 0, n_rows - 1)
    rb = np.clip(np.floor((lo_y - gy) / py).astype(int), 0, n_rows - 1)
    counts = (rb - ra + 1) * (cb - ca + 1)
    idx = np.repeat(np.arange(len(gdf)), counts)
    offset = np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)
    span = np.repeat(cb - ca + 1, counts)
    rr = np.repeat(ra, counts) + offset // span
    cc = np.repeat(ca, counts) + offset % span
    boxes = shapely.box(gx + cc * px, gy + (rr + 1) * py, gx + (cc + 1) * px, gy + rr * py)
    keep = shapely.intersects(gdf.geometry.to_numpy()[idx], boxes)
    return idx[keep], (rr * n_cols + cc)[keep]


records = []
for segment, name in PARISHES:
    store = zarr.open_group(
        FsspecStore.from_url(f"{BUCKET}/Barbados_IBF/Barbados/fim_store_{segment}_v1.zarr"), mode="r")
    store_attrs = dict(store.attrs)

    parts = []
    for type_, (layer, columns) in LAYERS.items():
        part = pyogrio.read_dataframe(source, layer=layer, columns=columns,
                                      where=f"ADM1_PCODE = '{segment.split('_')[0]}'", use_arrow=True)
        part["type"] = type_
        parts.append(part)
    gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
    for column in ["population_per_building", "building_area_m2", "road_length_m"]:
        gdf[column] = gdf[column].fillna(0.0)
    gdf = gdf.to_crs(store_attrs["crs"])

    idx, cells = pair_to_grid(gdf, store_attrs)
    grid = np.asarray(store["depth"][STORM], dtype="float32").ravel()
    grid = np.where(np.isfinite(grid) & (grid > 0), grid, 0.0)
    per_feature = np.zeros(len(gdf), dtype="float32")
    np.maximum.at(per_feature, idx, grid[cells])

    hit = gdf[per_feature >= store_attrs["extent_threshold_m"]]
    everyone = gdf.population_per_building.sum()
    records.append({
        "parish": name,
        "rain_mm": pd.read_csv(
            f"{BUCKET}/Barbados_IBF/Barbados/fim_store_{segment}_v1.zarr/index.csv"
        ).magnitude_mm[STORM],
        "buildings": int((hit["type"] == "building").sum()),
        "population": hit.population_per_building.sum(),
        "roads_km": hit.road_length_m.sum() / 1000,
        "share": 100 * hit.population_per_building.sum() / everyone,
    })

island = pd.DataFrame(records).set_index("parish")
island.round(1)

In [ ]:
# Two views of the same eleven rows, side by side, because they disagree -- and
# the disagreement is the interesting part.
ranked_count = island.sort_values("population")
ranked_share = island.sort_values("share")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].barh(ranked_count.index, ranked_count.population, color="#4c78a8")
axes[0].set_xlabel("people in flooded buildings")
axes[0].set_title("How many people — absolute")

bars = axes[1].barh(ranked_share.index, ranked_share.share, color="#e45756")
axes[1].set_xlabel("% of the parish's own population")
axes[1].set_title("How many people — as a share of the parish")
for bar, value in zip(bars, ranked_share.share):
    axes[1].text(value, bar.get_y() + bar.get_height() / 2, f" {value:.0f}%", va="center", fontsize=8)

fig.suptitle(f"Storm {STORM} across Barbados: the same scenario, eleven parishes")
fig.tight_layout(); plt.show()

**Read the two panels against each other.** Saint Michael dominates the absolute count
because that is where the people are, but ranked by *share of its own population* Christ
Church comes out on top instead — and the two smallest parishes swap as well. Past that,
the two orderings are nearly the same, and that agreement is itself the finding: storm
150 puts between 13% and 47% of every parish's population in a flooded building. It is a
whole-island event, so there is no parish the national count is hiding.

Do not assume that holds. Run the same cell for a storm at the dry end of the library and
the two panels come apart, because a small scenario floods a few places rather than
everywhere. **Which denominator you divide by is a decision made in the chart, not in the
model**, and no plugin can guess which one the person looking at the screen needs.

Other things the same dataframe would answer without new data — `subtype` splits
buildings into residential, commercial, education, medical and more; `building_area_m2`
gives floor area rather than counts; sweeping `STORM` from 1 to 199 turns any of these
charts into a curve against rainfall.

## 9. From notebook to dashboard

Now put it on a screen. **[Exercise 2](../../docs/Barbados/exercise_2.rst)** builds the
`Barbados Hands On 2` dashboard, and every tile in it is a section of this notebook
wrapped in a plugin:

| notebook section | dashboard tile | plugin |
|---|---|---|
| 1 and 2 — the store, one storm, its magnitude and extent | the headline card | `uffis_storm_card_barbados` |
| 3 to 6 — features, sampling, bands, the table | the summary table | `uffis_storm_impact_summary_barbados` |
| 3 to 7 — the same features, returned as GeoJSON | the impact map layer | `uffis_storm_impact_layer_barbados` |
| the depth raster of section 1 | the depth image layer | read straight from the store by the map |

A **plugin** is an installable Python class that TethysDash discovers on its own, through
an entry point in `pyproject.toml`:

```toml
[project.entry-points."intake.drivers"]
uffis_storm_impact_summary_barbados = "tgf_wmo_plugins.storm_impact_summary:StormImpactSummaryBarbados"
uffis_storm_impact_layer_barbados   = "tgf_wmo_plugins.storm_impact_layer:StormImpactLayerBarbados"
uffis_storm_card_barbados           = "tgf_wmo_plugins.storm_card:StormCardBarbados"
```

The class itself is mostly declaration. `type` picks the frontend renderer — and
therefore dictates the shape `run()` must return — and `args` is what the dashboard
author fills in:

```python
class StormImpactSummaryBarbados(BaseStormImpactSummaryUnit):
    LANG = "en"
    country = "barbados"
    unit_arg = "parish"                       # section 3's ADM1_PCODE filter
    default_unit = "BB08_SaintMichael"
    type = "table"                            # -> run() returns {"title", "data": [rows]}
    args = {"parish": BARBADOS_PARISH_OPTIONS,   # a list of {"value","label"} -> a dropdown
            "index": UNIT_STORM_OPTIONS}         # storm positions 0-199 -> a dropdown
    name = "uffis_storm_impact_summary_barbados"  # must match the entry-point key
```

**The two arguments are the two things this notebook hard-coded.** `PARISH` in section 0
becomes the `parish` argument; `STORM` in section 1 becomes `index`. In the dashboard
each is bound to a **variable input** — `${Parish}` and `${Storm}` — so moving a dropdown
re-runs the plugin with a new value. That is the whole mechanism: a notebook constant
becomes a plugin argument becomes a control on the screen.

The two dropdowns do not carry their own lists of choices, either. Each takes its options
straight from the plugin argument it feeds, by naming it
`Flood Maps (English): Storm Impact Summary (Barbados) - Parish`. So the eleven parishes
are declared **once**, in the plugin's `BARBADOS_PARISH_OPTIONS`, and the control cannot
drift out of sync with what consumes it — add a parish to the plugin and the dropdown
grows by itself. This is the practical reason argument names are a published contract: a
dashboard stores that string, so renaming `parish` breaks every control bound to it.

**Map layers answer two questions, so they implement two methods.** `run()` fires once,
when the author adds the layer, and returns the layer's *scaffold* — name, style rules,
legend. `fetch_features()` fires on every change to a bound variable and returns the
GeoJSON. Splitting them means changing the storm re-runs only the part that depends on
the storm.

```python
class StormImpactLayerBarbados(BaseStormImpactLayerUnit):
    type = "map_layer"
    dynamic_map_layer = True                 # this flag is what enables fetch_features()
    args = {"parish": BARBADOS_PARISH_OPTIONS, "index": UNIT_STORM_OPTIONS}

    def run(self):                           # CONFIGURE time: style and legend, once
        builder = LayerConfigurationBuilder("Flooded buildings and roads", "GeoJSON")
        builder.set_plugin_source(self.name, self.received_args)
        ...
        return builder.build()

    def fetch_features(self):                # RUN time: section 7's features, as GeoJSON
        self.send_update("Sampling depth...", percentage_complete=40)
        flooded = unit_banded_features(self.country, self.unit(), self.get_arg("index", 150))
        return {"type": "FeatureCollection", "features": [...],
                "crs": {"type": "name", "properties": {"name": "EPSG:4326"}}}
```

Two details from this notebook that the plugin has to get right: the (feature, cell)
pairing of section 4 is cached **per parish**, because it is expensive and
storm-independent but changes the moment the parish does; and only affected features are
returned, because section 7's payload is re-fetched on every storm change.

**If you want your section 8 charts on a dashboard**, the route is the same. A `plotly`
plugin returns a figure dict instead of a table, and the rest of the class is what you
see above. Start from `notebooks/01_plugin_example.ipynb`, which builds the simplest
possible plugin from nothing.

### Things to try

- Change `STORM` and re-run. Storms 0 and 1 are dry; 199 is the largest in the library at
  1,038 mm. Watch the impact table and the critical-facility chart move together.
- Change `PARISH` to another of the eleven and re-run the whole notebook. Nothing else has
  to change — the store, the receptor filter and the population denominator all follow
  from it. This is exactly what the dashboard's parish dropdown does.
- Section 4 takes the **maximum** depth over each footprint. Try the mean instead
  (`np.add.at` plus a count) and see how the table changes. Which is the more defensible
  number to hand a decision-maker?
- Add a band at `1.5 m or more` to `DEPTH_BANDS` and re-run. Now try adding one at
  `3 m or more`. What happens, and why is that a property of the data rather than the code?
- Build a fourth chart in section 8. `subtype` and `building_area_m2` are both sitting in
  the dataframe untouched. What would you show a mayor that the three existing charts do not?
- Take the section 8 parish sweep and run it for three different storms instead of one.
  Does the ranking of the parishes stay the same as the storm grows?